In [ ]:
from __future__ import annotations

from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

ARTIFACT_DIR = Path("artifacts")

TRAIN_PATH = ARTIFACT_DIR / "03_train.csv"

EDA_DIR = ARTIFACT_DIR / "04_eda"
CHART_DIR = EDA_DIR / "charts"

FINDINGS_PATH = EDA_DIR / "04_eda_findings.md"
MISSING_PATH = EDA_DIR / "04_missing_summary.csv"
NUMERICAL_PATH = EDA_DIR / "04_numerical_summary.csv"
CATEGORICAL_PATH = EDA_DIR / "04_categorical_summary.csv"
FEATURE_SIGNAL_PATH = EDA_DIR / "04_feature_signal_summary.csv"

EDA_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. VERIFY INPUT
# ============================================================

assert TRAIN_PATH.exists(), (
    f"Training artifact not found: {TRAIN_PATH}"
)

print(f"Reading TRAIN only: {TRAIN_PATH}")


# ============================================================
# 3. LOAD TRAIN DATA
# ============================================================

df = pd.read_csv(
    TRAIN_PATH,
    low_memory=False,
)

print("\nTRAIN DATA")
print("=" * 90)

print(f"Rows:    {len(df):,}")
print(f"Columns: {df.shape[1]:,}")

assert "is_late" in df.columns
assert df["order_id"].is_unique

print("Unique order_id: PASS")


# ============================================================
# 4. PARSE DATE COLUMNS
# ============================================================

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

date_columns = [
    column
    for column in date_columns
    if column in df.columns
]

for column in date_columns:
    df[column] = pd.to_datetime(
        df[column],
        format="mixed",
        errors="coerce",
    )

print("\nDATE COLUMNS")
print("=" * 90)
print(date_columns)


# ZIP code is a location identifier, not a continuous numeric feature
if "customer_zip_code_prefix" in df.columns:
    df["customer_zip_code_prefix"] = (
        df["customer_zip_code_prefix"]
        .astype("string")
    )


# ============================================================
# 5. SHAPE, DTYPES, MEMORY
# ============================================================

print("\nDATA TYPES")
print("=" * 90)

dtype_df = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(dtype) for dtype in df.dtypes],
    "non_null": df.notna().sum().values,
    "unique_values": [
        df[column].nunique(dropna=False)
        for column in df.columns
    ],
})

display(dtype_df)

memory_mb = (
    df.memory_usage(deep=True).sum()
    / 1024**2
)

print(f"\nMemory usage: {memory_mb:.2f} MB")


# ============================================================
# 6. COLUMN GROUPS
# ============================================================

id_columns = [
    column
    for column in df.columns
    if (
        column.endswith("_id")
        or column == "order_id"
    )
]

numeric_columns = (
    df.select_dtypes(
        include=["number"]
    )
    .columns
    .tolist()
)

categorical_columns = (
    df.select_dtypes(
        include=["object", "string", "category"]
    )
    .columns
    .tolist()
)

numeric_columns = [
    column
    for column in numeric_columns
    if column != "is_late"
]

categorical_columns = [
    column
    for column in categorical_columns
    if column not in id_columns
]

print("\nCOLUMN GROUPS")
print("=" * 90)

print(f"Numeric:     {len(numeric_columns)}")
print(f"Categorical: {len(categorical_columns)}")
print(f"Dates:       {len(date_columns)}")
print(f"IDs:         {len(id_columns)}")


# ============================================================
# 7. MISSING VALUES
# ============================================================

missing_df = (
    df.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missing_df["missing_percentage"] = (
    missing_df["missing_count"]
    / len(df)
    * 100
)

missing_df = (
    missing_df[
        missing_df["missing_count"] > 0
    ]
    .sort_values(
        "missing_percentage",
        ascending=False,
    )
)

print("\nMISSING VALUES")
print("=" * 90)

display(missing_df)

missing_df.to_csv(
    MISSING_PATH
)


# ============================================================
# 8. TARGET DISTRIBUTION
# ============================================================

target_distribution = (
    df["is_late"]
    .value_counts()
    .sort_index()
    .rename_axis("is_late")
    .reset_index(name="count")
)

target_distribution["percentage"] = (
    target_distribution["count"]
    / len(df)
    * 100
)

print("\nTARGET DISTRIBUTION")
print("=" * 90)

display(target_distribution)


fig, ax = plt.subplots(figsize=(7, 5))

ax.bar(
    ["On time / early", "Late"],
    [
        (df["is_late"] == 0).sum(),
        (df["is_late"] == 1).sum(),
    ],
)

ax.set_title("Training target distribution")
ax.set_ylabel("Orders")

fig.tight_layout()

target_chart_path = (
    CHART_DIR
    / "target_distribution.png"
)

fig.savefig(
    target_chart_path,
    dpi=150,
    bbox_inches="tight",
)

plt.show()


# ============================================================
# 9. NUMERICAL SUMMARY
# ============================================================

usable_numeric = [
    column
    for column in numeric_columns
    if column not in [
        "delivery_delay_days",
    ]
]

numerical_summary = (
    df[usable_numeric]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
    .T
)

numerical_summary["missing"] = (
    df[usable_numeric]
    .isna()
    .sum()
)

numerical_summary["skew"] = (
    df[usable_numeric]
    .skew(
        numeric_only=True
    )
)

print("\nNUMERICAL SUMMARY")
print("=" * 90)

display(numerical_summary)

numerical_summary.to_csv(
    NUMERICAL_PATH
)


# ============================================================
# 10. NUMERICAL RELATION WITH TARGET
# ============================================================

signal_rows = []

for column in usable_numeric:

    if df[column].notna().sum() == 0:
        continue

    grouped = (
        df.groupby(
            "is_late"
        )[column]
        .agg(
            [
                "count",
                "mean",
                "median",
            ]
        )
    )

    on_time_mean = (
        grouped.loc[0, "mean"]
        if 0 in grouped.index
        else np.nan
    )

    late_mean = (
        grouped.loc[1, "mean"]
        if 1 in grouped.index
        else np.nan
    )

    signal_rows.append({
        "feature": column,
        "on_time_mean": on_time_mean,
        "late_mean": late_mean,
        "absolute_mean_difference": (
            abs(
                late_mean
                - on_time_mean
            )
            if pd.notna(late_mean)
            and pd.notna(on_time_mean)
            else np.nan
        ),
    })

numeric_signal_df = (
    pd.DataFrame(signal_rows)
    .sort_values(
        "absolute_mean_difference",
        ascending=False,
    )
)

print("\nNUMERICAL FEATURES VS LABEL")
print("=" * 90)

display(
    numeric_signal_df.head(20)
)


# ============================================================
# 11. CORRELATIONS
# ============================================================

correlation_columns = [
    column
    for column in usable_numeric
    if df[column].nunique(
        dropna=True
    ) > 1
]

correlation_df = (
    df[
        correlation_columns
        + ["is_late"]
    ]
    .corr(
        numeric_only=True
    )["is_late"]
    .drop("is_late")
    .sort_values(
        key=abs,
        ascending=False,
    )
    .rename(
        "correlation_with_is_late"
    )
    .to_frame()
)

print("\nNUMERICAL CORRELATION WITH LABEL")
print("=" * 90)

display(
    correlation_df.head(20)
)


# ============================================================
# 12. CATEGORICAL SUMMARY
# ============================================================

categorical_rows = []

for column in categorical_columns:

    categorical_rows.append({
        "column": column,
        "unique_values": (
            df[column]
            .nunique(
                dropna=True
            )
        ),
        "missing_count": (
            df[column]
            .isna()
            .sum()
        ),
        "missing_percentage": (
            df[column]
            .isna()
            .mean()
            * 100
        ),
        "most_common_value": (
            df[column]
            .mode(
                dropna=True
            )
            .iloc[0]
            if df[column]
            .notna()
            .any()
            else None
        ),
    })

categorical_summary_df = (
    pd.DataFrame(
        categorical_rows
    )
    .sort_values(
        "unique_values",
        ascending=False,
    )
)

print("\nCATEGORICAL SUMMARY")
print("=" * 90)

display(
    categorical_summary_df
)

categorical_summary_df.to_csv(
    CATEGORICAL_PATH,
    index=False,
)


# ============================================================
# 13. SELECTED CATEGORICAL FEATURES VS LABEL
# ============================================================

candidate_categorical = [
    "customer_state",
    "payment_types",
    "seller_states",
    "product_categories",
]

candidate_categorical = [
    column
    for column in candidate_categorical
    if column in df.columns
]

categorical_signal_frames = []

for column in candidate_categorical:

    temp = (
        df.groupby(
            column,
            dropna=False,
        )
        .agg(
            orders=(
                "order_id",
                "count",
            ),
            late_rate=(
                "is_late",
                "mean",
            ),
        )
        .reset_index()
    )

    temp["late_percentage"] = (
        temp["late_rate"]
        * 100
    )

    temp["feature"] = column

    temp = temp[
        temp["orders"] >= 100
    ]

    temp = temp.sort_values(
        "late_percentage",
        ascending=False,
    )

    categorical_signal_frames.append(
        temp
    )

    print(
        f"\n{column.upper()} VS LABEL"
    )
    print("=" * 90)

    display(
        temp.head(15)
    )


if categorical_signal_frames:

    categorical_signal_df = (
        pd.concat(
            categorical_signal_frames,
            ignore_index=True,
        )
    )

else:
    categorical_signal_df = (
        pd.DataFrame()
    )


# ============================================================
# 14. CUSTOMER STATE ANALYSIS
# ============================================================

if "customer_state" in df.columns:

    state_analysis = (
        df.groupby(
            "customer_state"
        )
        .agg(
            orders=(
                "order_id",
                "count",
            ),
            late_orders=(
                "is_late",
                "sum",
            ),
            late_rate=(
                "is_late",
                "mean",
            ),
        )
        .reset_index()
    )

    state_analysis[
        "late_percentage"
    ] = (
        state_analysis[
            "late_rate"
        ]
        * 100
    )

    state_analysis = (
        state_analysis[
            state_analysis["orders"]
            >= 100
        ]
        .sort_values(
            "late_percentage",
            ascending=False,
        )
    )

    print("\nCUSTOMER STATE LATE RATE")
    print("=" * 90)

    display(
        state_analysis.head(15)
    )


    top_states = (
        state_analysis
        .head(15)
        .sort_values(
            "late_percentage"
        )
    )

    fig, ax = plt.subplots(
        figsize=(9, 7)
    )

    ax.barh(
        top_states[
            "customer_state"
        ],
        top_states[
            "late_percentage"
        ],
    )

    ax.set_title(
        "Late-delivery rate by customer state"
    )

    ax.set_xlabel(
        "Late orders (%)"
    )

    fig.tight_layout()

    state_chart_path = (
        CHART_DIR
        / "late_rate_by_customer_state.png"
    )

    fig.savefig(
        state_chart_path,
        dpi=150,
        bbox_inches="tight",
    )

    plt.show()


# ============================================================
# 15. DATE FEATURES FOR EDA ONLY
# ============================================================

df["purchase_year"] = (
    df[
        "order_purchase_timestamp"
    ]
    .dt.year
)

df["purchase_month"] = (
    df[
        "order_purchase_timestamp"
    ]
    .dt.month
)

df["purchase_weekday"] = (
    df[
        "order_purchase_timestamp"
    ]
    .dt.day_name()
)

df["purchase_hour"] = (
    df[
        "order_purchase_timestamp"
    ]
    .dt.hour
)


# ============================================================
# 16. MONTHLY LATE RATE
# ============================================================

df["purchase_year_month"] = (
    df[
        "order_purchase_timestamp"
    ]
    .dt.to_period("M")
    .astype(str)
)

monthly_df = (
    df.groupby(
        "purchase_year_month"
    )
    .agg(
        orders=(
            "order_id",
            "count",
        ),
        late_rate=(
            "is_late",
            "mean",
        ),
    )
    .reset_index()
)

monthly_df[
    "late_percentage"
] = (
    monthly_df[
        "late_rate"
    ]
    * 100
)

print("\nMONTHLY LATE RATE")
print("=" * 90)

display(
    monthly_df
)


fig, ax = plt.subplots(
    figsize=(12, 6)
)

ax.plot(
    monthly_df[
        "purchase_year_month"
    ],
    monthly_df[
        "late_percentage"
    ],
    marker="o",
)

ax.set_title(
    "Late-delivery rate over time"
)

ax.set_xlabel(
    "Purchase month"
)

ax.set_ylabel(
    "Late orders (%)"
)

ax.tick_params(
    axis="x",
    rotation=45,
)

fig.tight_layout()

monthly_chart_path = (
    CHART_DIR
    / "monthly_late_rate.png"
)

fig.savefig(
    monthly_chart_path,
    dpi=150,
    bbox_inches="tight",
)

plt.show()


# ============================================================
# 17. WEEKDAY ANALYSIS
# ============================================================

weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

weekday_df = (
    df.groupby(
        "purchase_weekday"
    )
    .agg(
        orders=(
            "order_id",
            "count",
        ),
        late_rate=(
            "is_late",
            "mean",
        ),
    )
    .reindex(
        weekday_order
    )
    .reset_index()
)

weekday_df[
    "late_percentage"
] = (
    weekday_df[
        "late_rate"
    ]
    * 100
)

print("\nWEEKDAY ANALYSIS")
print("=" * 90)

display(
    weekday_df
)


fig, ax = plt.subplots(
    figsize=(9, 5)
)

ax.bar(
    weekday_df[
        "purchase_weekday"
    ],
    weekday_df[
        "late_percentage"
    ],
)

ax.set_title(
    "Late-delivery rate by purchase weekday"
)

ax.set_ylabel(
    "Late orders (%)"
)

ax.tick_params(
    axis="x",
    rotation=30,
)

fig.tight_layout()

weekday_chart_path = (
    CHART_DIR
    / "weekday_late_rate.png"
)

fig.savefig(
    weekday_chart_path,
    dpi=150,
    bbox_inches="tight",
)

plt.show()


# ============================================================
# 18. SELECTED NUMERICAL DISTRIBUTIONS
# ============================================================

distribution_features = [
    "item_count",
    "total_item_price",
    "total_freight_value",
    "total_payment_value",
    "avg_product_weight_g",
]

distribution_features = [
    column
    for column in distribution_features
    if column in df.columns
]

for column in distribution_features:

    values = (
        df[column]
        .dropna()
    )

    if len(values) == 0:
        continue

    upper_limit = (
        values.quantile(
            0.99
        )
    )

    plot_values = (
        values[
            values
            <= upper_limit
        ]
    )

    fig, ax = plt.subplots(
        figsize=(8, 5)
    )

    ax.hist(
        plot_values,
        bins=40,
    )

    ax.set_title(
        f"{column} distribution "
        "(up to 99th percentile)"
    )

    ax.set_xlabel(
        column
    )

    ax.set_ylabel(
        "Orders"
    )

    fig.tight_layout()

    chart_path = (
        CHART_DIR
        / f"{column}_distribution.png"
    )

    fig.savefig(
        chart_path,
        dpi=150,
        bbox_inches="tight",
    )

    plt.show()


# ============================================================
# 19. OUTLIER SUMMARY USING IQR
# ============================================================

outlier_rows = []

for column in usable_numeric:

    series = (
        df[column]
        .dropna()
    )

    if len(series) == 0:
        continue

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = (
        (series < lower)
        | (series > upper)
    ).sum()

    outlier_rows.append({
        "feature": column,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": int(
            outliers
        ),
        "outlier_percentage": (
            outliers
            / len(series)
            * 100
        ),
    })

outlier_df = (
    pd.DataFrame(
        outlier_rows
    )
    .sort_values(
        "outlier_percentage",
        ascending=False,
    )
)

print("\nIQR OUTLIER SUMMARY")
print("=" * 90)

display(
    outlier_df.head(20)
)


# ============================================================
# 20. LEAKAGE / FEATURE AVAILABILITY REVIEW
# ============================================================

obvious_leakage_columns = [
    "order_delivered_customer_date",
    "delivery_delay_days",
]

possible_prediction_time_columns = [
    "order_status",
    "order_approved_at",
    "order_delivered_carrier_date",
]

print("\nFEATURE AVAILABILITY / LEAKAGE REVIEW")
print("=" * 90)

print(
    "Obvious leakage columns:"
)

for column in obvious_leakage_columns:
    if column in df.columns:
        print(f"  - {column}")

print(
    "\nColumns requiring prediction-time decision:"
)

for column in possible_prediction_time_columns:
    if column in df.columns:
        print(f"  - {column}")

print(
    "\nThese must be finalized before Notebook 5 "
    "based on when prediction is triggered."
)


# ============================================================
# 21. BUILD FEATURE SIGNAL SUMMARY
# ============================================================

feature_signal_summary = (
    numeric_signal_df.copy()
)

feature_signal_summary[
    "correlation_with_is_late"
] = (
    correlation_df[
        "correlation_with_is_late"
    ]
    .reindex(
        feature_signal_summary[
            "feature"
        ]
    )
    .values
)

feature_signal_summary.to_csv(
    FEATURE_SIGNAL_PATH,
    index=False,
)


# ============================================================
# 22. AUTO-GENERATE FINDINGS
# ============================================================

train_late_rate = (
    df["is_late"]
    .mean()
    * 100
)

top_missing = (
    missing_df
    .head(10)
)

top_numeric_signals = (
    feature_signal_summary
    .head(10)
)

findings_lines = [
    "# Notebook 4 EDA Findings",
    "",
    "## Dataset",
    f"- Training rows: {len(df):,}",
    f"- Columns: {df.shape[1]:,}",
    f"- Memory usage: {memory_mb:.2f} MB",
    f"- Training late-delivery rate: {train_late_rate:.2f}%",
    "",
    "## Important observations",
    "- EDA was performed on the training split only.",
    "- The target is imbalanced, so accuracy alone should not be the main evaluation metric.",
    "- Missing values must be handled after understanding whether missingness carries business meaning.",
    "- High-cardinality categorical columns require careful encoding.",
    "- Numerical outliers should not be removed automatically without business justification.",
    "- Temporal changes in late-delivery rate should be considered because the split is chronological.",
    "",
    "## Leakage",
    "- order_delivered_customer_date must not be used as a prediction feature.",
    "- delivery_delay_days must not be used as a prediction feature.",
    "- order_status, order_approved_at, and order_delivered_carrier_date depend on the final prediction trigger and require a business decision before feature engineering.",
    "",
    "## Top missing columns",
]

for column, row in top_missing.iterrows():

    findings_lines.append(
        f"- {column}: "
        f"{int(row['missing_count']):,} "
        f"({row['missing_percentage']:.2f}%)"
    )


findings_lines.extend([
    "",
    "## Numerical signals to investigate",
])

for _, row in top_numeric_signals.iterrows():

    findings_lines.append(
        f"- {row['feature']}: "
        f"mean difference={row['absolute_mean_difference']:.4f}, "
        f"correlation={row['correlation_with_is_late']:.4f}"
    )


findings_lines.extend([
    "",
    "## Next step",
    "- Use these findings in Notebook 5 to decide feature transformations and the final prediction-time feature set.",
])


FINDINGS_PATH.write_text(
    "\n".join(
        findings_lines
    ),
    encoding="utf-8",
)


# ============================================================
# 23. VERIFY ARTIFACTS
# ============================================================

required_files = [
    FINDINGS_PATH,
    MISSING_PATH,
    NUMERICAL_PATH,
    CATEGORICAL_PATH,
    FEATURE_SIGNAL_PATH,
    target_chart_path,
    monthly_chart_path,
    weekday_chart_path,
]

for path in required_files:

    assert path.exists(), (
        f"Missing artifact: {path}"
    )

    assert path.stat().st_size > 0, (
        f"Empty artifact: {path}"
    )


# ============================================================
# 24. FINAL RESULT
# ============================================================

print("\n")
print("=" * 90)
print("NOTEBOOK 4 RESULT")
print("=" * 90)

print("SUCCESS")

print(
    "Detailed EDA was completed "
    "using TRAIN only."
)

print(
    "Missing values were analyzed."
)

print(
    "Numerical and categorical features "
    "were inspected."
)

print(
    "Relationships with the target "
    "were analyzed."
)

print(
    "Temporal and geographic patterns "
    "were inspected."
)

print(
    "Potential leakage columns "
    "were documented."
)

print(
    f"Findings saved to: {FINDINGS_PATH}"
)

print(
    f"Charts saved to: {CHART_DIR}"
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Environment OK")